# 🐬 Colab 免费 GPU · 无审查模型服务

把 Dolphin / abliterated 模型跑在免费 T4 上，经 Cloudflare 隧道暴露为 **OpenAI 兼容端点**，opencode 直接接入（两台机器都能用）。

**步骤**
1. 菜单：代码执行程序 → 更改运行时类型 → **T4 GPU**
2. 代码执行程序 → 全部运行
3. 「⑤ 保活」cell 保持运行、笔记本别关
4. 把最后输出的 `OPENAI_BASE_URL` 和 `MODEL` 发给 opencode，助手会自动写进两台机器配置

**限制**：免费 T4 ≈16GB 显存；单会话约 12h 上限，空闲 ~90 分钟断开；每周有 GPU 配额；每次新会话 VM 是全新的，模型需重下（几分钟）。

**安全**：隧道 URL 公开可达（无鉴权），拿到就能调模型——不要外传，断开重开会换新 URL。

**合规**：仅用于合法用途，内容责任自负。

In [ ]:
# ① 确认 GPU 已启用（应显示 T4 / L4，不能是 CPU）
!nvidia-smi

In [ ]:
#@title ② 选择模型与参数（默认 = 拒答率最低的 dolphincoder-15B）
MODEL = "dolphincoder:15b" #@param ["dolphincoder:15b", "dolphincoder", "dolphin-llama3", "dolphin3:8b", "maternion/spark-x2.5-heretic:4b", "hf.co/BoldingBuilds/Abliterated-MiMo-V2.6-Distill-Qwen-9B-GGUF:Q4_K_M"]
CONTEXT_LEN = 16384 #@param {type:"integer"}
print("model =", MODEL)
print("context =", CONTEXT_LEN)

In [ ]:
#@title ③ 安装 Ollama 并启动服务
import os, time, urllib.request
os.environ["OLLAMA_CONTEXT_LENGTH"] = str(CONTEXT_LEN)
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
rc = os.system("curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama-install.log 2>&1")
if rc != 0:
    print(open("/tmp/ollama-install.log").read()[-2000:])
    raise RuntimeError("ollama 安装失败")
os.system("nohup ollama serve > /tmp/ollama.log 2>&1 &")
for _ in range(30):
    time.sleep(1)
    try:
        ver = urllib.request.urlopen("http://localhost:11434/api/version", timeout=2).read().decode()
        print("ollama up:", ver)
        break
    except Exception:
        pass
else:
    print(open("/tmp/ollama.log").read()[-2000:])
    raise RuntimeError("ollama 未启动")

In [ ]:
import os
#@title ④ 下载模型并自测（首次约 2~8 分钟，看网速）
import json, urllib.request
rc = os.system(f"ollama pull {MODEL}")
if rc != 0:
    raise RuntimeError("模型下载失败，检查 MODEL 名称")
req = urllib.request.Request(
    "http://localhost:11434/v1/chat/completions",
    data=json.dumps({
        "model": MODEL,
        "messages": [{"role": "user", "content": "Reply with exactly: OK"}],
        "max_tokens": 20,
        "think": False,
    }).encode(),
    headers={"Content-Type": "application/json"},
)
resp = json.loads(urllib.request.urlopen(req, timeout=180).read())
print("自测回复:", resp["choices"][0]["message"]["content"][:200])

In [ ]:
import os
#@title ⑤ 启动 Cloudflare 隧道 → 输出接入地址
import re, subprocess, time
if not os.path.exists("/tmp/cloudflared"):
    rc = os.system("curl -L -o /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared")
    if rc != 0:
        raise RuntimeError("cloudflared 下载失败")
subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=open("/tmp/cf.log", "w"),
    stderr=subprocess.STDOUT,
)
url = ""
for _ in range(30):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/cf.log").read())
    if m:
        url = m.group(0)
        break
if not url:
    print(open("/tmp/cf.log").read()[-2000:])
    raise RuntimeError("隧道启动失败")
BASE_URL = url.rstrip("/") + "/v1"
print("=" * 60)
print("OPENAI_BASE_URL =", BASE_URL)
print("MODEL           =", MODEL)
print("CONTEXT         =", CONTEXT_LEN)
print("=" * 60)
print("把上面三行发给 opencode 即可自动接入")

In [ ]:
#@title ⑥ 保活（保持运行，笔记本别关；停止请按 ■ 中断）
import time, urllib.request
print("keepalive started — 每 60s 心跳一次")
while True:
    time.sleep(60)
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=5)
        print(time.strftime("%H:%M:%S"), "alive  ✅")
    except Exception as e:
        print("服务掉线:", e)
        break

### 接入 opencode（两种方式）

**方式 A（推荐）**：把 `OPENAI_BASE_URL` 和 `MODEL` 发到 opencode 对话里，助手会写进本机 + Codespace 两台的 `opencode.jsonc`。

**方式 B（手动）**：在 `opencode.jsonc` 的 `provider` 里加：

```jsonc
"colab": {
  "npm": "@ai-sdk/openai-compatible",
  "name": "Colab-Free",
  "options": {
    "baseURL": "https://xxxx.trycloudflare.com/v1",
    "apiKey": "ollama"
  },
  "models": {
    "dolphincoder:15b": { "name": "DolphinCoder-15B (Colab)", "limit": { "context": 16384, "output": 4096 } }
  }
}
```

之后在 `/models` 里选 `Colab-Free / dolphincoder:15b` 即可。

### 结束
会话到期/主动关闭时：跑下面的停止 cell，或直接点「停止并释放会话」。下次使用重新 Run all（模型会重下）。

In [ ]:
import os
#@title ⏹ 停止服务（可选）
os.system("pkill -f cloudflared")
os.system("pkill -f 'ollama serve'")
print("stopped — 隧道 URL 已失效")